# Environment Setup

This notebook verifies your OpenShift environment, deploys models on RHOAI, and tests InferenceService endpoints.

## 1. Verify Cluster Access

In [ ]:
import subprocess
import json

print("=" * 50)
print("OpenShift Environment Verification")
print("=" * 50)

checks = [
    ("oc CLI", ["oc", "version", "--client", "-o", "json"]),
    ("Cluster login", ["oc", "whoami"]),
    ("Cluster URL", ["oc", "whoami", "--show-server"]),
]

for name, cmd in checks:
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        output = result.stdout.strip()
        if result.returncode == 0:
            print(f"✅ {name}: {output[:80]}")
        else:
            print(f"❌ {name}: {result.stderr.strip()[:80]}")
    except FileNotFoundError:
        print(f"❌ {name}: command not found")

## 2. Verify RHOAI Operator

In [ ]:
%%bash
echo "=== RHOAI Operator ==="
oc get csv -n redhat-ods-operator 2>/dev/null | grep -i rhods || echo "⚠️  RHOAI operator not found"

echo ""
echo "=== GPU Nodes ==="
oc get nodes -l nvidia.com/gpu.present=true --no-headers 2>/dev/null || echo "⚠️  No GPU nodes labeled"

echo ""
echo "=== KServe/ModelMesh ==="
oc get crd inferenceservices.serving.kserve.io --no-headers 2>/dev/null && echo "✅ KServe CRD available" || echo "❌ KServe not installed"

## 3. Configure Environment Variables

In [ ]:
from pathlib import Path

env_path = Path("../.env")
sample_path = Path("../sample.env")

if not env_path.exists():
    if sample_path.exists():
        env_path.write_text(sample_path.read_text())
        print(f"Created {env_path} from sample.env")
        print("⚠️  Edit .env and fill in your tokens before proceeding.")
    else:
        print("❌ sample.env not found.")
else:
    print(f"✅ {env_path} already exists.")

## 4. Deploy Models on RHOAI

Deploy Qwen2.5-Coder models (FP8-quantized by RedHat AI) via vLLM InferenceServices.

In [ ]:
%%bash
echo "Applying RHOAI model manifests..."
oc apply -f manifests/00-rhoai-models.yaml

echo ""
echo "⏳ Waiting for InferenceServices to be ready (this may take several minutes)..."
echo "   Models need to download weights from Hugging Face on first deployment."
echo ""
echo "Monitor progress with:"
echo "  oc get inferenceservice -n rhoai-models -w"
echo "  oc get pods -n rhoai-models"

In [ ]:
%%bash
echo "=== InferenceService Status ==="
oc get inferenceservice -n rhoai-models

echo ""
echo "=== Pods ==="
oc get pods -n rhoai-models

## 5. Test Model Endpoints

Once InferenceServices show `READY=True`, test the endpoints.

In [ ]:
import subprocess
import json

# Get SA token for authentication
token_result = subprocess.run(
    ["oc", "create", "token", "default", "-n", "rhoai-models", "--duration=1h"],
    capture_output=True, text=True
)
SA_TOKEN = token_result.stdout.strip()

# Get InferenceService URLs
isvc_result = subprocess.run(
    ["oc", "get", "inferenceservice", "-n", "rhoai-models",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.status.url}\n{end}"],
    capture_output=True, text=True
)

print("RHOAI Model Endpoints:")
print("=" * 60)

endpoints = {}
for line in isvc_result.stdout.strip().split("\n"):
    if "=" in line:
        name, url = line.split("=", 1)
        endpoints[name] = url
        print(f"  {name}: {url}")

# Test first available endpoint
if endpoints:
    test_name = list(endpoints.keys())[0]
    test_url = endpoints[test_name]
    print(f"\nTesting {test_name}...")
    
    test_result = subprocess.run(
        ["curl", "-sk", f"{test_url}/v1/models",
         "-H", f"Authorization: Bearer {SA_TOKEN}"],
        capture_output=True, text=True, timeout=15
    )
    
    if test_result.returncode == 0:
        models = json.loads(test_result.stdout)
        print(f"✅ {test_name} responding. Available models: {[m['id'] for m in models.get('data', [])]}")
    else:
        print(f"❌ {test_name} not responding yet. Check pod status.")
else:
    print("⚠️  No InferenceService URLs found. Wait for deployment to complete.")

In [ ]:
%%bash
# Quick inference test
ISVC_URL=$(oc get inferenceservice qwen-coder-7b -n rhoai-models -o jsonpath='{.status.url}' 2>/dev/null)
TOKEN=$(oc create token default -n rhoai-models --duration=1h)

if [ -z "$ISVC_URL" ]; then
    echo "⚠️  qwen-coder-7b not ready yet."
    exit 0
fi

echo "Testing inference on qwen-coder-7b..."
curl -sk "${ISVC_URL}/v1/chat/completions" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "qwen-coder-7b",
    "messages": [{"role": "user", "content": "Say hello in one word."}],
    "max_tokens": 10
  }' | python3 -m json.tool

## 6. Verify MaaS Availability

Check if MaaS (Models as a Service) is available on your cluster.

In [ ]:
%%bash
echo "=== MaaS Gateway ==="
kubectl get gateway -n openshift-ingress maas-default-gateway 2>/dev/null && echo "✅ MaaS Gateway found" || echo "⚠️  MaaS Gateway not found — install MaaS via RHOAI operator"

echo ""
echo "=== MaaS Endpoint ==="
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}' 2>/dev/null)
if [ -n "$CLUSTER_DOMAIN" ]; then
    echo "  https://maas.${CLUSTER_DOMAIN}"
else
    echo "⚠️  Could not determine cluster domain"
fi

## Next Steps

Once models are ready (`oc get inferenceservice -n rhoai-models` shows `READY=True`):

1. **Phase 1** → `1_mcp_servers/2_deploy_mcp_servers.ipynb` to deploy MCP tool servers
2. **Phase 2** → `2_ai_gateway/2_enable_maas.ipynb` to enable Models as a Service